In [1]:
import os
import shutil
import numpy as np
from PIL import Image
import cv2

In [2]:
# Modify to your own paths

# Dataset base directory
base_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Drishti-GS'

# Images original directory
train_images_original_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Drishti-GS/Training/Images'
test_images_original_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Drishti-GS/Test/Images'

# Masks original directory
train_masks_original_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Drishti-GS/Training/GT'
test_masks_original_dir = '/home/zhenyi/Project/Dundee/FunduSegmenter/Code/datasets/Drishti-GS/Test/Test_GT'

In [3]:
# Create target directory
train_images_dir = os.path.join(base_dir, 'training', 'images')
train_masks_dir = os.path.join(base_dir, 'training', 'masks')
test_images_dir = os.path.join(base_dir, 'testing', 'images')
test_masks_dir = os.path.join(base_dir, 'testing', 'masks')
os.makedirs(train_images_dir, exist_ok=True)
os.makedirs(train_masks_dir, exist_ok=True)
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_masks_dir, exist_ok=True)

In [4]:
# Save images to target directory
def process_images(original_dir, target_dir):
    for foldername in os.listdir(original_dir):
        for filename in os.listdir(os.path.join(original_dir, foldername)):
            shutil.copy(os.path.join(original_dir, foldername, filename), os.path.join(target_dir, filename))

process_images(train_images_original_dir, train_images_dir)
process_images(test_images_original_dir, test_images_dir)              

In [5]:
# Get the average masks, modify masks to required form and save to target directory
def process_masks(original_dir, target_dir):
    for foldername in os.listdir(original_dir):
        if os.path.isdir(os.path.join(original_dir, foldername, 'SoftMap')):
            savename = foldername
            together_list = []
            for filename in os.listdir(os.path.join(original_dir, foldername, 'SoftMap')):
            
                old_mask = np.asarray(Image.open(os.path.join(original_dir, foldername, 'SoftMap', filename)))

                # Threshold indicates agreement among at least three experts, follow the setting from original paper
                # Available at: https://cdn.iiit.ac.in/cdn/cvit.iiit.ac.in/images/ConferencePapers/2015/Arunava2015AComprehensive.pdf
                _, new_mask = cv2.threshold(old_mask, int(255*0.75)-1, 255, cv2.THRESH_BINARY)
                together_list.append(new_mask)
            together = together_list[0].astype(np.uint16) + together_list[1].astype(np.uint16)
            together[together == 255] = 128
            together[together == 0] = 255
            together[together == 510] = 0

            together = Image.fromarray(together.astype(np.uint8))
            target_path = os.path.join(target_dir, savename+'.png')
            together.save(target_path, quality=100)

process_masks(train_masks_original_dir, train_masks_dir)
process_masks(test_masks_original_dir, test_masks_dir)